# Honeypot Corpus Generation — standalone

Builds the **synthetic honeypot training corpus** (`data/honeypot_final.log`) from the WEB-IDS23
flow CSVs, using [`scripts/webids23_to_honeypot_log_v9.py`](../scripts/webids23_to_honeypot_log_v9.py).

This is **Stage 2** of `end_to_end_pipeline.ipynb`, extracted so you can regenerate the corpus,
tune the generation parameters, and audit the result without running the whole study.

## ⚠️ Read this before you run it

**Regenerating the corpus invalidates the committed models.** A new log means new session-grouped
train/val/test splits, so `models/*.pkl` no longer correspond to it. After running this you
**must** retrain (Stage 5 of the main notebook, or `python scripts/18_train_stacked.py`), and
every downstream number in your thesis changes.

If you only want to *inspect* the existing corpus, skip to **Step 4 (Audit)** — it works on the
committed log without regenerating anything.

## What this generator actually does

The CSV supplies the **envelope**; the payloads are **synthesised**.

It reads only 6 columns — `uid, ts, id.orig_h, id.resp_h, service, attack_type` — for timing,
addresses and protocol. Everything else in the output is generated:

| Class | How it is built |
|---|---|
| **SQLi** | 7 generator families: `tautology`, `union_based`, `time_based_blind`, `boolean_blind`, `error_based`, `stacked_query`, `auth_bypass` |
| **XSS** | 3 families: `reflected`, `stored`, `dom_based` |
| **Benign** | Stateful sessions: each picks one browsing archetype + one identity, then advances coherently (search → browse → product → checkout) |

**Class labels come from the flag you pass a file under, not from the data.** A file given to
`--sqli-http` gets SQLi payloads and `label=1` regardless of what its `attack_type` column says.

## Four correctness properties (v5+)

1. **Enforced payload uniqueness.** v4 had 72.9 % duplicate rows because some families had a ~12-
   output component space sampled 36 k times. v9 expands the pools *and* enforces uniqueness
   against a run-wide seen-set (25 retries, then a nonce). Rows needing the nonce are flagged
   `forced_unique_nonce`.
2. **Two distinct duplicate flags.** `synthetic_duplicate` = the same flow row sampled twice
   (flow-level). `forced_unique_nonce` = payload-level collision fallback. Do not conflate them.
3. **User-agent realism.** v4 had 6 UAs, one literally `sqlmap/1.7.11` — a model given UA as a
   feature would learn `UA==sqlmap → malicious`. v9 uses ~18 weighted UAs.
4. **Schema noise removed.** `host` is always `ip:port`.

---
## Step 1 — Locate the repo and check the sources

Finds `dashboard/` (clones on Colab if needed) and fingerprints the five WEB-IDS23 CSVs.

In [ ]:
# ==========================================================================
# STEP 1 - LOCATE REPO + FINGERPRINT SOURCES
# ==========================================================================
import os, sys, json, time, hashlib, subprocess, collections
from pathlib import Path
from datetime import datetime, timezone

try:                                   # payloads contain non-cp1252 characters
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

IS_COLAB = ("google.colab" in sys.modules) or os.path.isdir("/content")
REPO_URL = "https://github.com/Judge09/AI-GIS_dashboard.git"

def find_root():
    for base in [Path.cwd()] + list(Path.cwd().parents):
        if (base / "dashboard" / "scripts" / "webids23_to_honeypot_log_v9.py").exists():
            return base / "dashboard"
        if (base / "scripts" / "webids23_to_honeypot_log_v9.py").exists():
            return base
    return None

ROOT = find_root()
if ROOT is None and IS_COLAB:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL,
                    "/content/AI-GIS_dashboard"], check=True)
    ROOT = Path("/content/AI-GIS_dashboard/dashboard")
if ROOT is None:
    raise FileNotFoundError(
        "Could not find scripts/webids23_to_honeypot_log_v9.py. Run this notebook "
        "from inside the repo, or set ROOT by hand.")

ROOT    = ROOT.resolve()
SCRIPTS = ROOT / "scripts"
DATA    = ROOT / "data"
PREP    = DATA / "prepared"
os.chdir(ROOT)
print("ROOT =", ROOT)

GEN = SCRIPTS / "webids23_to_honeypot_log_v9.py"

# The five sources live in data/prepared/ in this repo (NOT data/).
SRC = {
    "--sqli-http":  PREP / "web-ids23_sql_injection_http (3).csv",
    "--sqli-https": PREP / "web-ids23_sql_injection_https (2).csv",
    "--xss-http":   PREP / "web-ids23_xss_http (1).csv",
    "--xss-https":  PREP / "web-ids23_xss_https (1).csv",
    "--benign":     PREP / "web-ids23_benign (1).csv",
}

def fp(p):
    """size + sha256 prefix, so you can prove which file a run consumed."""
    if not p.exists():
        return "[MISSING]"
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return "%8.1f MB  sha256=%s" % (p.stat().st_size / 1048576.0, h.hexdigest()[:16])

print("\nGENERATOR")
print("   %-14s %s" % ("script", fp(GEN)))
print("\nSOURCE CSVs (data/prepared/)")
missing = []
for flag, p in SRC.items():
    print("   %-14s %s" % (flag, fp(p)))
    if not p.exists():
        missing.append(p.name)
if missing:
    print("\n   MISSING: %s" % missing)
    print("   These are NOT committed to the repo (271 MB total). Copy them into")
    print("   data/prepared/ before generating, or skip to Step 4 to audit the")
    print("   existing committed log instead.")
else:
    print("\n   All five sources present.")

---
## Step 2 — Parameters, and the target arithmetic

The knobs, and what they do:

| Parameter | Default | Effect |
|---|---|---|
| `SEED` | 42 | Deterministic output. Same seed + same inputs = byte-identical log. |
| `OBFUSCATE_PROB` | 0.4 | Fraction of attack payloads passed through the obfuscator. Your committed log shows **71 %** of attack rows obfuscated — the obfuscator also fires inside some generators. |
| `STRATEGY` | `match_min` | How class targets are set — see below |
| `BENIGN_RATIO` | 1.0 | `benign_target = ratio × (sqli_target + xss_target)` |

### The strategy choice matters more than it looks

- **`match_min`** — both classes get `min(sqli_natural, xss_natural)`. Balanced, but the smaller
  class is the binding constraint.
- **`match_max`** — both get the larger count; the smaller class is heavily oversampled (flows
  reused, flagged `synthetic_duplicate`).
- **`custom`** — you set `--custom-sqli-count` / `--custom-xss-count` explicitly.

With your files: SQLi has **176,884** flows, XSS only **9,091**. Under `match_min` both become
9,091 — so **~168 k SQLi flows go unused**. That is the correct default (balance beats volume),
but it is worth knowing you are discarding 95 % of your SQLi source.

The cell below prints the exact arithmetic *before* you commit to a run.

In [ ]:
# ==========================================================================
# STEP 2 - PARAMETERS + TARGET ARITHMETIC (prints; generates nothing)
# ==========================================================================
SEED           = 42
OBFUSCATE_PROB = 0.4
STRATEGY       = "match_min"     # match_min | match_max | custom
BENIGN_RATIO   = 1.0
CUSTOM_SQLI    = None            # only used when STRATEGY == "custom"
CUSTOM_XSS     = None

OUT_LOG = DATA / "honeypot_final.log"

import pandas as pd

def count_rows(p):
    """Chunked count - these files are up to 217 MB."""
    return sum(len(c) for c in pd.read_csv(p, usecols=["uid"], chunksize=200_000))

counts = {}
if not missing:
    print("SOURCE FLOW COUNTS")
    for flag, p in SRC.items():
        counts[flag] = count_rows(p)
        print("   %-14s %8d rows" % (flag, counts[flag]))

    sqli_nat = counts["--sqli-http"] + counts["--sqli-https"]
    xss_nat  = counts["--xss-http"]  + counts["--xss-https"]
    ben_nat  = counts["--benign"]

    if STRATEGY == "match_min":
        sqli_t = xss_t = min(sqli_nat, xss_nat)
    elif STRATEGY == "match_max":
        sqli_t = xss_t = max(sqli_nat, xss_nat)
    else:
        sqli_t = CUSTOM_SQLI if CUSTOM_SQLI is not None else sqli_nat
        xss_t  = CUSTOM_XSS  if CUSTOM_XSS  is not None else xss_nat
    ben_t = round(BENIGN_RATIO * (sqli_t + xss_t))

    print("\nTARGETS  (strategy=%s, benign_ratio=%s)" % (STRATEGY, BENIGN_RATIO))
    print("   SQLi natural %d -> target %d" % (sqli_nat, sqli_t))
    print("   XSS  natural %d -> target %d" % (xss_nat, xss_t))
    print("   benign avail %d -> target %d  (%.1f%% of pool)"
          % (ben_nat, ben_t, 100.0 * ben_t / max(ben_nat, 1)))
    print("   TOTAL ROWS   %d" % (sqli_t + xss_t + ben_t))

    if ben_t > ben_nat:
        print("\n   WARNING: benign target exceeds the pool - flows will be "
              "oversampled (reused, flagged synthetic_duplicate).")
    binding = "XSS" if xss_nat < sqli_nat else "SQLi"
    unused  = max(sqli_nat, xss_nat) - min(sqli_nat, xss_nat)
    if STRATEGY == "match_min" and unused:
        print("\n   NOTE: %s is the binding constraint; %d %s flows go UNUSED."
              % (binding, unused, "SQLi" if binding == "XSS" else "XSS"))

    # benign service composition - affects envelope realism, not labels
    bser = pd.read_csv(SRC["--benign"], usecols=["service"])
    http_n = int((bser["service"] == "http").sum())
    print("\n   Benign service mix: http=%d (%.1f%%), other/NaN=%d"
          % (http_n, 100.0 * http_n / len(bser), len(bser) - http_n))
    print("   The generator does NOT filter on service - it only tests")
    print("   service=='http' to pick port 80 vs 443. So dns/ssl/smtp flows")
    print("   become HTTPS-looking web requests. This costs envelope realism,")
    print("   NOT label correctness. Pre-filter to service=='http' if you want")
    print("   genuinely web-shaped benign envelopes (%d rows available)." % http_n)
else:
    print("Sources missing - cannot compute targets. Skip to Step 4 to audit the "
          "existing log.")

---
## Step 3 — Generate ⟨off by default⟩

Set `RUN_GENERATION = True` to actually build the corpus. It is off by default because this
**overwrites `data/honeypot_final.log`** and invalidates the committed models.

The existing log is backed up first, so a run is never destructive.

> **Note:** `import resource` in the generator is Unix-only. It is now guarded, so the script
> runs on Windows too — the only loss is the peak-memory line in the final report.

In [ ]:
# ==========================================================================
# STEP 3 - GENERATE THE CORPUS (off by default)
# ==========================================================================
RUN_GENERATION = True      # <-- set True to regenerate

if not RUN_GENERATION:
    print("SKIPPED: RUN_GENERATION is False.")
    print("   The committed data/honeypot_final.log is left untouched.")
    print("   Set RUN_GENERATION = True to rebuild it, then RETRAIN:")
    print("      python scripts/18_train_stacked.py --epochs 6 --seed 42")
elif missing:
    print("SKIPPED: source CSVs missing: %s" % missing)
else:
    import shutil
    if OUT_LOG.exists():
        stamp  = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        backup = OUT_LOG.with_suffix(".log.bak_%s" % stamp)
        shutil.copy2(OUT_LOG, backup)
        print("Backed up existing log -> %s\n" % backup.name)

    cmd = [sys.executable, str(GEN)]
    for flag, p in SRC.items():
        cmd += [flag, str(p)]
    cmd += ["--strategy", STRATEGY,
            "--benign-ratio", str(BENIGN_RATIO),
            "--seed", str(SEED),
            "--obfuscate-prob", str(OBFUSCATE_PROB),
            "-o", str(OUT_LOG)]
    if STRATEGY == "custom":
        if CUSTOM_SQLI is not None: cmd += ["--custom-sqli-count", str(CUSTOM_SQLI)]
        if CUSTOM_XSS  is not None: cmd += ["--custom-xss-count",  str(CUSTOM_XSS)]

    print("$ %s\n" % " ".join(cmd))
    t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, errors="replace")
    for line in proc.stdout:
        print("   | " + line.rstrip())
    proc.wait()
    print("\n-> exit=%s in %.1fs" % (proc.returncode, time.time() - t0))
    if proc.returncode != 0:
        raise RuntimeError("generator failed - see output above")
    print("Wrote %s (%.1f MB)" % (OUT_LOG, OUT_LOG.stat().st_size / 1048576.0))
    print("\nNEXT: retrain, or every downstream number is stale:")
    print("   python scripts/18_train_stacked.py --epochs 6 --seed 42")

---
## Step 4 — Audit the corpus

Works whether or not you regenerated. This is the ground truth for anything you claim about the
training data.

**What to look at, and what a bad value means:**

| Metric | Healthy | A bad value means |
|---|---|---|
| Label balance | ~50/50 | Class imbalance the model will exploit |
| Malformed lines | 0 | Partially-written or corrupted log |
| Schema keys | all 17 | Version mismatch — a v4 log in a v9 pipeline |
| Exact-duplicate rate | < 5 % | The v4 duplication bug is back; metrics inflated |
| `forced_unique_nonce` | < 5 % | A family's component space shrank |
| Sessions | thousands | Too few groups for a clean grouped split |
| Payload length p99 | < 200 | LSTM truncates at 200 chars |

In [ ]:
# ==========================================================================
# STEP 4 - AUDIT THE CORPUS
# ==========================================================================
EXPECTED_KEYS = {"time", "source_ip", "host", "method", "uri", "user_agent",
                 "request_body", "referer", "flow_uid", "dup_index", "session_id",
                 "session_seq", "label", "attack_family", "obfuscated",
                 "synthetic_duplicate", "forced_unique_nonce"}

if not OUT_LOG.exists():
    raise FileNotFoundError("%s not found - run Step 3, or restore it." % OUT_LOG)

rows, bad = [], []
with open(OUT_LOG, encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError as e:
            bad.append((i, str(e)))

n        = len(rows)
labels   = collections.Counter(r.get("label") for r in rows)
families = collections.Counter(r.get("attack_family") for r in rows)
sessions = {r.get("session_id") for r in rows}
keys     = set().union(*(set(r) for r in rows)) if rows else set()

print("FILE      %s  (%.1f MB)" % (OUT_LOG.name, OUT_LOG.stat().st_size / 1048576.0))
print("ROWS      %d parsed, %d malformed" % (n, len(bad)))
print("SESSIONS  %d  (avg %.1f rows/session)" % (len(sessions), n / max(len(sessions), 1)))

print("\nLABEL BALANCE")
for lab in sorted(labels, key=lambda x: (x is None, x)):
    print("   label=%-5s %7d  (%5.1f%%)" % (lab, labels[lab], 100.0 * labels[lab] / n))

print("\nATTACK FAMILY")
for fam, c in families.most_common():
    print("   %-20s %7d  (%5.1f%%)" % (fam, c, 100.0 * c / n))

payloads = ["%s %s" % (r.get("uri", ""), r.get("request_body", "") or "") for r in rows]
uniq     = len(set(payloads))
dup_pct  = 100.0 * (n - uniq) / n
nonce_n  = sum(1 for r in rows if r.get("forced_unique_nonce"))
syndup   = sum(1 for r in rows if r.get("synthetic_duplicate"))
atk      = [r for r in rows if r.get("label") == 1]
obf      = sum(1 for r in atk if r.get("obfuscated"))

print("\nUNIQUENESS  (the v5 correctness fixes)")
print("   unique uri+body ....... %d / %d  (duplicate rate %.2f%%)" % (uniq, n, dup_pct))
print("   forced_unique_nonce ... %d  (%.2f%% - payload-level fallback)"
      % (nonce_n, 100.0 * nonce_n / n))
print("   synthetic_duplicate ... %d  (flow-level oversampling)" % syndup)
print("   obfuscated attacks .... %d / %d  (%.1f%%)"
      % (obf, len(atk), 100.0 * obf / max(len(atk), 1)))

lens = sorted(len(p) for p in payloads)
pct  = lambda q: lens[min(int(q * len(lens)), len(lens) - 1)]
print("\nPAYLOAD LENGTH  (LSTM truncates at 200)")
print("   min/p50/p90/p99/max ... %d / %d / %d / %d / %d"
      % (lens[0], pct(.50), pct(.90), pct(.99), lens[-1]))
print("   over 200 chars ........ %d (%.1f%%)"
      % (sum(1 for L in lens if L > 200), 100.0 * sum(1 for L in lens if L > 200) / n))

print("\nCHECKS")
def chk(ok, msg):
    print("   [%s] %s" % ("PASS" if ok else "FAIL", msg))
chk(n > 0,                      "corpus is non-empty")
chk(not bad,                    "all lines parsed (%d malformed)" % len(bad))
chk(not (EXPECTED_KEYS - keys), "schema has all 17 keys")
chk(set(labels) <= {0, 1},      "labels strictly 0/1")
chk(min(labels.get(0,0), labels.get(1,0)) / max(labels.get(0,1), labels.get(1,1)) > 0.8,
                                "classes roughly balanced")
chk(dup_pct < 5.0,              "duplicate rate %.2f%% < 5%% (v4 was 72.9%%)" % dup_pct)
chk(100.0 * nonce_n / n < 5.0,  "nonce fallback %.2f%% < 5%%" % (100.0 * nonce_n / n))
chk(len(sessions) > 100,        "enough sessions (%d) for a grouped split" % len(sessions))
chk(pct(.99) <= 200,            "p99 length %d within the 200-char LSTM window" % pct(.99))

---
## Step 5 — Look at what it produced

Numbers do not tell you whether the payloads are *sensible*. Read a few.

Pay particular attention to the **benign hard negatives** — text engineered to look attack-shaped
without being an attack (`O'Brien's Hardware`, `x = a + b; y = c - d;`, `Error code: 500 -- see
logs`). That pool is your false-positive surface, and expanding it is the highest-value change
you can make to this generator.

In [ ]:
# ==========================================================================
# STEP 5 - SAMPLE THE OUTPUT
# ==========================================================================
import random, urllib.parse
random.seed(0)

def show(title, subset, k=6):
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)
    for r in random.sample(subset, min(k, len(subset))):
        q = r.get("uri", "")
        body = r.get("request_body", "") or ""
        raw = (q.split("?", 1)[1] if "?" in q else body)
        try:
            dec = urllib.parse.unquote_plus(raw)
        except Exception:
            dec = raw
        print("   [%s] %s %s" % (r.get("attack_family", "?"),
                                 r.get("method", ""), q.split("?", 1)[0]))
        # Always show the PAYLOAD: a POST keeps it in the body, so printing
        # only the URI hides exactly what you came here to inspect.
        print("        payload: %s" % (dec[:100] if dec else "(none)"))

by_fam = collections.defaultdict(list)
for r in rows:
    by_fam[r.get("attack_family")].append(r)

show("SQLi samples", [r for r in rows if r.get("label") == 1
                      and r.get("attack_family") in
                      {"tautology","union_based","time_based_blind","boolean_blind",
                       "error_based","stacked_query","auth_bypass"}], 8)
show("XSS samples",  [r for r in rows if r.get("attack_family") in
                      {"reflected","stored","dom_based"}], 6)
show("BENIGN samples - the false-positive surface",
     [r for r in rows if r.get("label") == 0], 10)

print("\n" + "=" * 72)
print("PER-FAMILY EXAMPLE (one of each)")
print("=" * 72)
for fam in sorted(by_fam):
    r = by_fam[fam][0]
    print("   %-18s %s" % (fam, r.get("uri", "")[:80]))

---
## Step 6 — What to do next

**If you regenerated the corpus, you are not finished.** The models on disk were trained on the
*old* log and no longer correspond to this one.

```bash
# 1. retrain (~11 min CPU, ~2 min T4)
python scripts/18_train_stacked.py --epochs 6 --seed 42

# 2. re-evaluate
python scripts/17_evaluate_csv.py --csv data/eval/llm_holdout_full.csv \
       --out reports/llm_holdout_results.json
python scripts/19_eval_by_type.py --csv data/eval/llm_holdout_full.csv
```

Or just run `end_to_end_pipeline.ipynb` / `run_pipeline.py`, which does all of it in order with
full instrumentation.

### Tuning ideas, roughly by value

| Change | Why it might help |
|---|---|
| **Expand `BENIGN_HARD_NEGATIVE_PHRASES`** (~40 entries today) | Directly reduces false positives — the highest-value change available |
| Pre-filter benign to `service == "http"` | Envelope realism: 7.2 % → 100 % genuinely web traffic, still 59 k rows |
| Raise `OBFUSCATE_PROB` | More evasion-resistant training, at the cost of realism |
| `--strategy custom` with a larger SQLi count | Uses more of the 168 k unused SQLi flows; costs class balance |

### Reproducibility

Same seed + same input files = byte-identical log. The audit in Step 4 prints the SHA-256 of
every source, so any log you produce can be traced back to exactly what made it.